# 05 - Modeling: Functional Connectivity (Power-264)

Compares multiple classifiers for MDD vs. HC using FC features.  
Feature selection (SelectKBest) is applied **inside** the cross-validation loop  
to prevent data leakage.

**Input:** data/processed/datasetConnectivityLabels.csv 
**Output:** results/metrics/fc_cv_results.csv + figures

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import RocCurveDisplay, ConfusionMatrixDisplay

RESULTS_DIR = "../results"
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/metrics",  exist_ok=True)

In [ ]:
import pandas as pd
import os

df = pd.read_csv("data/connectivity_power264.csv")        
df.to_parquet("data/connectivity_power264.parquet",
              engine="pyarrow",
              compression="snappy",  
              index=False)
print("Listo. Tamaño:")

print(f"  CSV     : {os.path.getsize('data/connectivity_power264.csv')     / 1e6:.1f} MB")
print(f"  Parquet : {os.path.getsize('data/connectivity_power264.parquet') / 1e6:.1f} MB")

Listo. Tamaño:
  CSV     : 1653.1 MB
  Parquet : 791.3 MB


In [ ]:
df = pd.read_parquet("data/connectivity_power264.parquet")

df["diagnosis"] = (
    df["subject_id"].str.extract(r"^S\d+-(\d+)-")
    .squeeze()
    .astype(int)
    .map({1: 1, 2: 0})
)

df["site"] = (
    df["subject_id"].str.extract(r"^(S\d+)-")
    .squeeze()
)

assert df["diagnosis"].notna().all(), "Hay subject_ids que no matchearon el patrón"
assert df["site"].notna().all(),      "Hay subject_ids sin site"

meta_cols    = {"subject_id", "diagnosis", "site"}
feature_cols = [c for c in df.columns if c not in meta_cols]

X      = df[feature_cols].values.astype(np.float32)  # (N, 34716)
y      = df["diagnosis"].values                       # (N,)  int
groups = df["site"].values                            # (N,)  str — para LOSO

print(f"X shape      : {X.shape}")
print(f"y shape      : {y.shape}  — MDD: {(y==1).sum()}  HC: {(y==0).sum()}")
print(f"Sites (LOSO) : {len(np.unique(groups))}")
print()

site_balance = (
    df.groupby("site")["diagnosis"]
    .value_counts()
    .unstack(fill_value=0)
    .rename(columns={1: "MDD", 0: "HC"})
)
site_balance["total"]   = site_balance["MDD"] + site_balance["HC"]
site_balance["mdd_pct"] = (site_balance["MDD"] / site_balance["total"] * 100).round(1)

print(site_balance.sort_values("total", ascending=False).to_string())
print()

single_class = site_balance[(site_balance["MDD"] == 0) | (site_balance["HC"] == 0)]
if not single_class.empty:
    print(f"{len(single_class)} sitio(s) con una sola clase - LOSO fallará en estos folds:")
    print(single_class.to_string())
else:
    print("Todos los sitios tienen ambas clases - LOSO viable")


X shape      : (2341, 34716)
y shape      : (2341,)  — MDD: 1249  HC: 1092
Sites (LOSO) : 24

diagnosis   HC  MDD  total  mdd_pct
site                               
S20        251  282    533     52.9
S21         70   86    156     55.1
S25         63   89    152     58.6
S8          75   75    150     50.0
S1          74   74    148     50.0
S15         50   50    100     50.0
S9          50   50    100     50.0
S14         32   64     96     66.7
S17         44   47     91     51.6
S7          49   38     87     43.7
S10         33   50     83     60.2
S3          37   27     64     42.2
S24         31   32     63     50.8
S23         30   32     62     51.6
S16         31   31     62     50.0
S11         29   32     61     52.5
S2          30   30     60     50.0
S22         20   30     50     60.0
S4          24   24     48     50.0
S13         17   25     42     59.5
S18         20   21     41     51.2
S12          6   32     38     84.2
S6          15   15     30     50.0
S5    

In [3]:
K_FEATURES = 500   # reducido: 1000 es excesivo para ~2380 sujetos
                   # regla de pulgar: n_features << n_samples
                   # con LOSO (~150 sujetos por fold de test) aún más importante

# Factory para evitar que los folds compartan el mismo objeto selector
selector = lambda: SelectKBest(score_func=f_classif, k=K_FEATURES)

models = {

    # ── SVM (RBF) ─────────────────────────────────────────────────────────────
    # Mejor candidato para FC: kernel RBF captura relaciones no lineales
    # entre conexiones. class_weight="balanced" compensa desbalance MDD/HC.
    "SVM (RBF)": Pipeline([
        ("scaler",   StandardScaler()),
        ("selector", selector()),
        ("clf",      SVC(
            kernel="rbf",
            C=1,                    # reducido de 10: menos riesgo de overfit
            gamma="scale",          # = 1/(n_features * X.var()), correcto
            class_weight="balanced",
            probability=True,       # necesario para ROC-AUC
            random_state=42,
            cache_size=500,         # MB de caché para CPU — acelera entrenamiento
        ))
    ]),

    # ── Logistic Regression ───────────────────────────────────────────────────
    # Baseline lineal interpretable. lbfgs es estable con features escaladas.
    # Añadido C=0.1 (más regularización) porque 500 features con ~2000 sujetos
    # sigue siendo un ratio features/sujetos alto.
    "Logistic Regression": Pipeline([
        ("scaler",   StandardScaler()),
        ("selector", selector()),
        ("clf",      LogisticRegression(
            C=0.1,
            solver="lbfgs",
            class_weight="balanced",
            max_iter=1000,
            random_state=42,
            n_jobs=-1,
        ))
    ]),

    # ── Random Forest ─────────────────────────────────────────────────────────
    # No necesita scaler (invariante a escala).
    # n_estimators reducido a 100 para viabilidad en CPU con LOSO.
    # max_features="sqrt" es el default correcto para clasificación.
    "Random Forest": Pipeline([
        ("selector", selector()),
        ("clf",      RandomForestClassifier(
            n_estimators=100,       # reducido de 200 para velocidad CPU
            max_features="sqrt",    # ~22 features por split: estándar RF
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,              # usa todos los cores disponibles
        ))
    ]),

    # ── Gradient Boosting ─────────────────────────────────────────────────────
    # GB es el más lento en CPU. Configuración conservadora pero funcional.
    # subsample<1.0 = Stochastic GB: más rápido y menos overfit.
    # Añadido scaler: GB es sensible a escala cuando hay features muy distintas.
    "Gradient Boosting": Pipeline([
        ("scaler",   StandardScaler()),
        ("selector", selector()),
        ("clf",      GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.05,     # reducido: más estable con menos árboles
            max_depth=2,            # reducido de 3: MDD es señal débil y ruidosa
            subsample=0.8,          # Stochastic GB — más rápido en CPU
            min_samples_leaf=20,    # evita splits en grupos muy pequeños
            random_state=42,
        ))
    ]),
}

# ── Verificación rápida antes de correr LOSO ─────────────────────────────────
print(f"Modelos configurados : {list(models.keys())}")
print(f"K features por fold  : {K_FEATURES}")
print(f"Features totales     : {X.shape[1]:,}")
print(f"Sujetos              : {X.shape[0]}")
print(f"MDD / HC             : {(y==1).sum()} / {(y==0).sum()}")
print(f"Sites (folds LOSO)   : {len(set(groups))}")
print()

# Estimación de tiempo (muy aproximada para CPU)
n_folds = len(set(groups))
print(f"Estimación de folds LOSO : {n_folds}")
print("Orden de velocidad esperado:")
print("  Logistic Regression < SVM < Random Forest < Gradient Boosting")

NameError: name 'Pipeline' is not defined

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = ["accuracy", "roc_auc", "f1", "precision", "recall"]

rows = []
for name, pipe in models.items():
    print(f"Running {name}...")
    scores = cross_validate(pipe, X, y, cv=cv,
                            scoring=scoring, n_jobs=-1)
    rows.append({
        "Model":          name,
        "Accuracy":       np.mean(scores["test_accuracy"]),
        "Accuracy_std":   np.std( scores["test_accuracy"]),
        "ROC_AUC":        np.mean(scores["test_roc_auc"]),
        "ROC_AUC_std":    np.std( scores["test_roc_auc"]),
        "F1":             np.mean(scores["test_f1"]),
        "Precision":      np.mean(scores["test_precision"]),
        "Recall":         np.mean(scores["test_recall"]),
    })

results_df = pd.DataFrame(rows).sort_values("ROC_AUC", ascending=False)
results_df.to_csv(f"{RESULTS_DIR}/metrics/fc_cv_results.csv", index=False)
results_df.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
metrics = ["Accuracy", "ROC_AUC", "F1", "Precision", "Recall"]
x = np.arange(len(results_df))
width = 0.15

for i, m in enumerate(metrics):
    ax.bar(x + i * width, results_df[m], width, label=m)

ax.set_xticks(x + width * 2)
ax.set_xticklabels(results_df["Model"], rotation=15, ha="right")
ax.set_ylim(0.4, 0.85)
ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8, label="Chance")
ax.set_title("5-Fold CV — FC Power-264 features")
ax.legend(bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/figures/fc_model_comparison.png", dpi=150)
plt.show()

## Best model - detailed evaluation

In [ ]:
from sklearn.model_selection import train_test_split

best_name = results_df.iloc[0]["Model"]
best_pipe = models[best_name]
print(f"Best model: {best_name}")

# Single hold-out split for confusion matrix and ROC curve
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
best_pipe.fit(X_train, y_train)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ConfusionMatrixDisplay.from_estimator(
    best_pipe, X_test, y_test,
    display_labels=["HC", "MDD"],
    ax=axes[0], colorbar=False
)
axes[0].set_title(f"Confusion Matrix — {best_name}")

RocCurveDisplay.from_estimator(
    best_pipe, X_test, y_test, ax=axes[1]
)
axes[1].set_title(f"ROC Curve — {best_name}")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/figures/fc_best_model_eval.png", dpi=150)
plt.show()